# SDP-Meta Lab 2: CDC - Handling Update Events In Your Data Streams

In the first lab, `LAB 1 - Deploying and Onboarding new Pipelines`, you should have set up the SDP-Meta environment and deployed your first pipeline, which was an APPEND pipline with all records from the Bronze table being appended to the Silver table.

In this lab, we will walk through adding an additional pipeline which is a MERGE pipeline. In a merge pipeline, change events such as updates and deletes can be merged into your downstream tables.



#Initialize the Notebook

Run the cell below to populate the properties from the overrides we set up in Lab 1.

If this command fails, rerun Step 3 from the first lab. 

In [0]:
import json, os

# load the overrides file into a variable
overrides_path = "../.databricks/bundle/dev/variable-overrides.json"
with open(overrides_path, "r") as f:
    variable_overrides = json.load(f)

print(json.dumps(variable_overrides, indent=2))


## Configure the Onboarding files

For this lab, we want to update the Silver table so that only the most recent status is reported for each server, identified by the field `system_id`.
We can use the SDP features for supporting tables as SCD type 1 or type 2.

Supported SCD types:

- **Type 1** (scd_type: 1) — overwrite the existing row; no history retained
- **Type 2** (scd_type: 2) — retain full row history with validity timestamps


To configure SCD type on a table use the following settings:
- bronze_cdc_apply_changes - add SCD type configuration to the bronze table
- silver_cdc_apply_changes - add SCD type configuration to the silver table

These configuration objects contain the following fields:

| Field | Type | Required | Description |
|---|---|---|---|
| `keys` | array of strings | Yes | Primary key columns |
| `sequence_by` | string | Yes | Column(s) used to order events and determine the most recent |
| `scd_type` | string | Yes | `1` for overwrite or `2` for history |
| `apply_as_deletes` | string | No | SQL expression identifying delete events |
| `except_column_list` | array of strings | No | Columns to exclude from the target table |
| `track_history_column_list` | array of strings | No | (Type 2 only) Columns whose changes trigger a new history row |
| `track_history_except_column_list` | array of strings | No | (Type 2 only) Columns to exclude from history tracking |





#Task 1 - Update the pipeline to merge change entries on the Silver table

1. Update the Onboarding.json document to include the following settings:

```json
    "silver_cdc_apply_changes": {
      "keys": ["system_id"],
      "sequence_by": "event_time",
      "scd_type": "1"
    }
```

2. Deploy the bundle to copy the Onboarding.json document to the deploy location

3. Run the Onboarding job to update the dataflowspec tables

4. Run a Full Refresh on the silver table in the Pipeline
   a. Click on the Pipeline in Bundle Resources section of the Deployments tab
   b. Click on the Silver table to select it
   c. Click on the kabob menu (three stacked dots icon) to expand the run options
   d. Click on `Run with full refresh` - This will refresh just the silver table without rerunning the entire pipeline

![image_1788976616301.png](./img/image_1788976616301.png "image_1788976616301.png")

5. Once you have updated, deployed, and ran the pipeline, validate the changes below.

In [0]:
#validate the bronze and silver tables
from pyspark.sql import functions as F

uc_catalog_name = variable_overrides["uc_catalog_name"]
sdp_meta_schema = variable_overrides["sdp_meta_schema"]

bronze_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.host_status_updates_bronze');

silver_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.host_status_updates_silver');

display(bronze_df)

display(silver_df)

duplicate_ids = silver_df.groupBy("system_id").count().filter(F.col("count") > 1)
assert duplicate_ids.count() == 0, f"Silver table should contain only one entry per system_id, but found duplicates: {duplicate_ids.collect()}"

print("Lab 2 Task 1 passed.")

# Task 2 - Update the Bronze table to Support SCD type 2

Follow the steps you configured for Task 1 to add the following setting to the Bronze table. 

```json
    "bronze_cdc_apply_changes": {
      "keys": ["system_id"],
      "sequence_by": "event_time",
      "scd_type": "2"
    }
```

Now, redeploy the bundle and run the onboarding job. Run a Full Refresh of the pipeline.

Once this is completed, check the bronze table and you will see that SCD type 2 columns have been added to the table: __START_AT and __END_AT.

In [0]:
uc_catalog_name = variable_overrides["uc_catalog_name"]
sdp_meta_schema = variable_overrides["sdp_meta_schema"]

bronze_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.host_status_updates_bronze');

display(bronze_df)

assert "__START_AT" in bronze_df.columns and "__END_AT" in bronze_df.columns, "Bronze table should contain SCD type 2 columns (__START_AT and __END_AT)"

print("Lab 2 Task 2 passed.")

#Task 3 - Add support for a deleting records
As the final step in this lab, we will add support for deleting records to our Silver table. We want to treat `TERMINATED` nodes as deletes and remove them from the silver table.
 
Add the following property to the `silver_cdc_apply_changes` onboarding field.
```json
    "silver_cdc_apply_changes": {
        ...
      "apply_as_deletes": "status = 'TERMINATED'"
    }
```

We should also clean up the duplicate entries in the bronze table:
```json
    "bronze_cdc_apply_changes": {
      ...
      "track_history_except_column_list": ["event_time"]
    }
```

Now deploy and refresh the pipeline.

In [0]:
uc_catalog_name = variable_overrides["uc_catalog_name"]
sdp_meta_schema = variable_overrides["sdp_meta_schema"]

bronze_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.host_status_updates_bronze');

silver_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.host_status_updates_silver');

display(bronze_df)

display(silver_df)

assert bronze_df.filter("system_id = 'SYS-2001'").select("status").count() == 5, "Expected 5 items for system_id = 'SYS-2001'"

assert silver_df.filter("system_id = 'SYS-2001'").count() == 0, "There should be no record with system_id = 'SYS-2001' in the silver table"

print("Lab 2 Task 3 passed.")

#Task 4 - Add metadata to the tables

We should add provenance metadata to the records to list the source files for the records.
We are going to add two columns, `source_file` and `source_file_date`, at the root level of the table.

Under `source_details` add a new section `source_metadata`.

`source_metadata` has three possible subfields:

- **include_autoloader_metadata_column**: When 'True', will include the `_metadata` column in the table
- **autoloader_metadata_col_name**: Renames the `_metadata` column
- **select_metadata_cols**: Will select specific columns from the `_metadata` and add them to the table

Since we are only interested in two columns from `_metadata` and want them at the root level of the table, the configuration to use is below:

```json
      "source_metadata": {
        "select_metadata_cols": {
            "source_file": "_metadata.file_name",
            "source_file_date": "_metadata.file_modification_time"
        }
      }
```



In [0]:
uc_catalog_name = variable_overrides["uc_catalog_name"]
sdp_meta_schema = variable_overrides["sdp_meta_schema"]

bronze_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.host_status_updates_bronze');

silver_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.host_status_updates_silver');

display(bronze_df)

display(silver_df)

assert "source_file" in bronze_df.columns, "Bronze table should contain 'source_file' column"
assert "source_file_date" in bronze_df.columns, "Bronze table should contain 'source_file_date' column"

print("Lab 2 Task 4 passed.")

#Task 5 - Enrich entries from a lookup table

The final task in this lab is to enrich the entries in the silver table by joining them to values from a lookup table.

Run the query below to create the lookup table and a lookup function.


In [0]:
uc_catalog_name = variable_overrides["uc_catalog_name"]
sdp_meta_schema = variable_overrides["sdp_meta_schema"]

# Create the lookup table with region, hostname, and service_group
spark.sql(f"""
CREATE OR REPLACE TABLE {uc_catalog_name}.{sdp_meta_schema}.service_group_lookup (
  region STRING,
  hostname STRING,
  service_group STRING
)
""")

# Populate the lookup table with sample data
spark.sql(f"""
INSERT INTO {uc_catalog_name}.{sdp_meta_schema}.service_group_lookup VALUES
  ('us-east', 'web-01', 'Web Services'),
  ('us-east', 'db-01', 'Database Services'),
  ('us-east', 'api-01', 'Application Services'),
  ('us-west', 'web-01', 'Web Services'),
  ('us-west', 'db-01', 'Database Services'),
  ('us-west', 'api-01', 'Application Services'),
  ('eu-west', 'web-01', 'Web Services'),
  ('eu-west', 'db-01', 'Database Services'),
  ('eu-west', 'api-01', 'Application Services'),
  ('ap-south', 'web-01', 'Web Services'),
  ('ap-south', 'db-01', 'Database Services'),
  ('ap-south', 'api-01', 'Application Services')
""")

print(f"Lookup table created: {uc_catalog_name}.{sdp_meta_schema}.service_group_lookup")

# Create a lookup function that returns the service group for a given region and hostname
spark.sql(f"""
CREATE OR REPLACE FUNCTION {uc_catalog_name}.{sdp_meta_schema}.get_service_group(region_val STRING, hostname_val STRING)
RETURNS STRING
RETURN (
  SELECT service_group
  FROM {uc_catalog_name}.{sdp_meta_schema}.service_group_lookup
  WHERE region = region_val AND hostname = hostname_val
  LIMIT 1
)
""")

print(f"Lookup function created: {uc_catalog_name}.{sdp_meta_schema}.get_service_group")

# Display the lookup table
display(spark.sql(f"SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.service_group_lookup"))

Now add the lookup values to the `host_status_updates_silver_transformations.json` file.


Transformations files has a select_exp entry for each table that will be invoked to transform the entries from the bronze table into the silver table. You can invoke functions as well as cast value using standard SQL expressions.



**Hint**: you will need to add a new row to the `select_exp` under the target table: `host_status_updates_csv_silver`.
This new row should invoke your lookup function and set the name of the value as `service_group`.

In [0]:
uc_catalog_name = variable_overrides["uc_catalog_name"]
sdp_meta_schema = variable_overrides["sdp_meta_schema"]

silver_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.host_status_updates_silver');

display(silver_df)

# assert
assert "service_group" in silver_df.columns, "Silver table should contain 'service_group' column"

assert silver_df.filter("system_id = 'SYS-2003'").select("service_group").first()[0] == "Database Services", \
    "Expected service_group for system_id 'SYS-2003' to be 'Database Services'"

print("Lab 2 Task 5 passed.")

#LAB 2 - BONUS

As a bonus task, add an expectation to the host_status_updates_silver_dqe.json file (in the conf/dqe folder) that will log a warning for any entry that doesnt have an assigned service_group value.


## Troubleshooting

| Symptom | Cause | Fix |
| --- | --- | --- |
| `It is not allowed to change SCD Type within the same pipeline. Please restart the pipeline with full refresh for catalog.schema.silver_table.` | Updated the CDC configuration for a pipeline and ran the pipeline without a full refresh | Run a Full Refresh on the selected table or pipeline |
| `Error: no value assigned to required variable sdp_meta_schema.` | No variable override found in  ../.databricks/bundle/dev/variable-overrides.json | Run Cell 6 in Step 3 to create and set the bundle variable overrides |
| `Table 'sdp_meta_lab.<schema>.host_status_updates_csv_bronze' has a user-specified schema that is incompatible with the schema inferred from its query.` | The defined schema does not match the expected schema after SCD transforms | Remove the setting `source_schema_path` from the `source_details` |
| `sdp_meta_dependency is still set to __SET_ME__` | Wheel variable not configured | Set `sdp_meta_dependency` in variables or overrides |
| `Table or view not found: bronze_dataflowspec` | Onboarding not run | Run `databricks bundle run onboarding` |
| `The schema `sdp_meta_lab.sdp_meta_USERNAME` cannot be found.` | Onboarding file is reset to default values | Run Lab 1 cell 6 to populate the onboarding values |
| Pipeline starts but produces no tables | Onboarding misconfiguration | Confirm that the onboarding file has been set up correctly for the lab, check Lab 1 cell 3 |
| `Cannot read file from workspace path` on serverless | conf files not staged to Volume | Run onboarding (the `stage_conf` task handles this) |